# Oscillator encoding and adiabatic preparation

Derive the Pauli operators for a position grid and implement the quadratic and quartic terms using gates. This accompanies Chapters 9 and 10, after [the Fourier-transform examples](../../04-qft-and-phase-estimation/QFT.ipynb). The initial harmonic state here is loaded from sampled Gaussian amplitudes. The [next notebook](tutorial.ipynb) prepares an even initial state and turns on the kinetic term. The final comparison was cleared after making its grid construction explicit and needs rerunning.

[Qiskit tutorial](../../../../python-docs/qiskit.md)


In [1]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit.primitives import StatevectorEstimator, StatevectorSampler
from qiskit.circuit import Parameter, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import QFTGate
import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg as la

# Jordan Lee Preskill encoding

Let $n_q$ be the number of qubits and $N=2^{n_q}$ the number of position samples. Qubit $r=0$ is the least significant bit. The position grid is

$$
x_j=-x_{\max}+j\delta_x,\qquad
\delta_x=\frac{2x_{\max}}{N-1},\qquad j=0,\ldots,N-1.
$$

Write $j=\sum_{r=0}^{n_q-1}2^r b_r$, where $b_r$ is zero or one. Since $Z_r\ket{j}=(1-2b_r)\ket{j}$,

$$
\sum_r2^rZ_r\ket{j}=[(N-1)-2j]\ket{j}.
$$

Multiplication by $-x_{\max}/(N-1)$ gives $x_j$, so

$$
\hat x=-\frac{x_{\max}}{N-1}\sum_{r=0}^{n_q-1}2^r Z_r.
$$

We use the half-shifted Fourier grid

$$
p_k=-\frac{\pi}{\delta_x}+\left(k+\frac12\right)\frac{2\pi}{N\delta_x}
=\frac{\pi}{N\delta_x}[2k-(N-1)].
$$

The same binary identity gives the diagonal operator in the Fourier register,

$$
P_{\rm diag}=-\frac{\pi}{N\delta_x}\sum_{r=0}^{n_q-1}2^rZ_r.
$$

The kinetic-energy operator on the position register is obtained by changing to this Fourier basis, applying $P_{\rm diag}^2/2$ and changing back. The next derivation specifies the phases of that basis change.


Let $a=x_{\max}/(N-1)$. Expanding the square and combining the terms with $i<j$ and $j<i$ gives

$$
x^2=a^2\left[\sum_{i=0}^{n_q-1}4^i I+2\sum_{i<j}2^{i+j}Z_iZ_j\right].
$$

The diagonal terms contribute the first, state-independent term. We retain this term in the operator identity even when its phase is omitted from an ordinary evolution circuit.


For $H_x=(1+m_{\rm ctr})x^2/2$, the coefficient multiplying $Z_iZ_j$ is $c_{ij}=(1+m_{\rm ctr})a^2 2^{i+j}$. Therefore `x2_term` applies

$$
R_{ZZ}(2c_{ij}\delta t)=\exp(-i c_{ij}\delta t\,Z_iZ_j)
$$

to each pair $i<j$.


The following function implements this potential-energy step, up to its state-independent phase.


In [96]:
def x2_term(qc, m_ctr, delta_t, x_max):
    qubits = qc.num_qubits
    x_prefactor = - x_max / (2 ** qubits - 1)
    delta_x = 2 * x_max / (2 ** qubits - 1)
    for i in range(qubits):
        for j in range(i + 1, qubits):
            qc.rzz(2 * 2 ** (i + j) * (1 + m_ctr) * x_prefactor ** 2 * delta_t, i, j)

The Qiskit Fourier gate has matrix elements

$$
F_{kj}=\frac{1}{\sqrt N}\exp\left(\frac{2\pi i jk}{N}\right).
$$

For the shifted grids above we use $S_{kj}=N^{-1/2}\exp(i p_kx_j)$. Here $p_kx_j$ is already the dimensionless phase in units with $\hbar=1$. Substituting both grids gives

$$
\begin{aligned}
p_kx_j
&=\frac{2\pi}{N}\left(j-\frac{N-1}{2}\right)
                    \left(k-\frac{N-1}{2}\right)\\
&=\frac{2\pi jk}{N}-\frac{\pi(N-1)}{N}(j+k)
  +\frac{\pi(N-1)^2}{2N}.
\end{aligned}
$$

Define $\alpha=\pi(1/N-1)$ and $D_{jj}=e^{i\alpha j}$. Then

$$
S=e^{i\pi(N-1)^2/(2N)}DFD.
$$

The constant phase cancels in $S^\dagger P_{\rm diag}^2S$. To implement $D$, apply $R_z(\alpha 2^r)$ to each qubit. Indeed,

$$
\prod_{r=0}^{n_q-1}R_z(\alpha2^r)\ket{j}
=e^{-i\alpha(N-1)/2}e^{i\alpha j}\ket{j}.
$$

The first factor is another state-independent phase. Thus the code applies these rotations, `QFTGate`, the same rotations, the diagonal kinetic evolution, and the inverse operations in reverse order. The positive sign in $S$ matches Qiskit's Fourier gate. Reversing that Fourier convention reverses the momentum labels and leaves the kinetic energy unchanged.


The momentum grid and its diagonal Pauli operator were derived above. Squaring that diagonal operator gives the same pairwise form as the potential term. The following function performs the Fourier basis change around these rotations.


In [97]:
def p2_term(qc, lam0f, delta_t, x_max):
    qubits = qc.num_qubits
    delta_x = 2 * x_max / (2 ** qubits - 1)
    p_prefactor = - np.pi / 2 ** qubits / delta_x
    alpha = (1 / 2 ** qubits - 1) * np.pi
    for i in range(qubits):
        qc.rz(alpha * 2 ** i, i)
    qc.append(QFTGate(qubits), range(qubits))
    for i in range(qubits):
        qc.rz(alpha * 2 ** i, i)
    for i in range(qubits):
        for j in range(i + 1, qubits):
            qc.rzz(2 * 2 ** (i + j) * lam0f * p_prefactor ** 2 * delta_t, i, j)
    for i in range(qubits):
        qc.rz(- alpha * 2 ** i, i)
    qc.append(QFTGate(qubits).inverse(), range(qubits))
    for i in range(qubits):
        qc.rz(- alpha * 2 ** i, i)

For the anharmonic oscillator,

$$
H=\frac{p^2}{2}+\frac{x^2}{2}+\lambda x^4.
$$

Let $a=x_{\max}/(N-1)$ and $w_i=2^i$. Since $x=-a\sum_i w_iZ_i$, expanding its fourth power means sorting four indices by their repetitions. Four distinct indices occur in $4!=24$ orders. A repeated pair and two distinct remaining indices occur in $4!/2!=12$ orders. Three equal indices and one different index occur in four orders. Four equal indices, or two equal pairs, give the identity because $Z_i^2=I$. Consequently

$$
\begin{aligned}
x^4=a^4\Bigg[&
\left(\sum_i w_i^4+6\sum_{i<j}w_i^2w_j^2\right)I\\
&+24\sum_{i<j<k<l}w_iw_jw_kw_lZ_iZ_jZ_kZ_l\\
&+12\sum_i\sum_{\substack{j<k\\j,k\ne i}}w_i^2w_jw_kZ_jZ_k
+4\sum_i\sum_{j\ne i}w_i^3w_jZ_iZ_j\Bigg].
\end{aligned}
$$

All indices run from zero to $n_q-1$. This accounts for every term, including the identity contribution. The circuit below omits that contribution during ordinary state evolution because it changes only the global phase. For a controlled evolution in phase estimation or a Hadamard test, the phase must be restored, or the associated energy offset must be included explicitly. Each retained coefficient $c$ produces a rotation of angle $2c\,\delta t$, since $R_{ZZ}(\theta)=\exp(-i\theta Z_iZ_j/2)$. The CNOT ladder in `rz4` implements the same rule for four Pauli $Z$ operators.


In [121]:
def rz4(qc, theta, i, j, k, l):
    qc.cx(i, j)
    qc.cx(j, k)
    qc.cx(k, l)
    qc.rz(theta, l)
    qc.cx(k, l)
    qc.cx(j, k)
    qc.cx(i, j) # I mistyped this as (k, l) during the tutorial

In [110]:
def x4_term(qc, lam, delta_t, x_max):
    qubits = qc.num_qubits
    x_prefactor = - x_max / (2 ** qubits - 1)
    delta_x = 2 * x_max / (2 ** qubits - 1)
    '''
    Apply the Z_i Z_j Z_k Z_l term: four site interaction
    '''
    for i in range(qubits):
        for j in range(i + 1, qubits):
            for k in range(j + 1, qubits):
                for l in range(k + 1, qubits):
                    coeff = 2 ** (i + j + k + l) * 24 * lam * x_prefactor ** 4
                    rz4(qc, 2 * coeff * delta_t, i, j, k, l)
    '''
    Apply the two-site piece with two and only two equal indices
    '''
    for i in range(qubits):
        for j in range(qubits):
            if j == i:
                continue
            for k in range(j + 1, qubits):
                if k == i:
                    continue
                coeff = 2 ** (2 * i + j + k) * 12 * lam * x_prefactor ** 4 # I wrote 2 * i  + j + l instead of k here in the video
                qc.rzz(2 * coeff * delta_t, j, k)
    '''
    Apply the two qubit interaction where three indices are equal
    '''
    for i in range(qubits):
        for j in range(qubits):
            if j == i:
                continue
            coeff = 2 ** (3 * i + j) * 4 * lam * x_prefactor ** 4
            qc.rzz(2 * coeff * delta_t, i, j)

In [111]:
def strang_step(qc, lam, m_ctr, delta_t, x_max):
    ''' Apply strang split exp(A+B) = exp(A/2) exp(B) exp(A/2)
    A here is x^2 + lambda * x^4 piece
    '''
    # apply x^2 + lam * x^4
    x2_term(qc, m_ctr, delta_t / 2, x_max)
    x4_term(qc, lam, delta_t / 2, x_max)
    # apply the p^2 piece
    p2_term(qc, 1, delta_t, x_max)
    # apply again the x^2 ... piece (this is A / 2)
    x2_term(qc, m_ctr, delta_t / 2, x_max)
    x4_term(qc, lam, delta_t / 2, x_max)

Now we need ramp terms
$$ r(u) = 3u^2 - 2 u^3 $$

In [112]:
# first thing is the ramp
def r(u):
    return u * u * (3.0 - 2.0 * u)

In [113]:
def lamOf(u, lam):
    if 0.0 <= u <= 1.0:
        return lam * r(u)
    if 1.0 < u <= 2:
        return lam
    return 0

# The chosen ramp reaches delta m^2 = -6 * lam, then returns to zero.
def dm2Of(u, lam):
    if 0.0 <= u <= 1.0:
        return - 6 * lam * r(u)
    if 1.0 < u <= 2:
        return - 6 * lam * (1 - r(u - 1))
    return 0.0

Need to build the full adiabatic evolution circuit

In [114]:
def build_circuit(lam, num_qubits, x_max, tPrep = 25.0, nSteps = 250):
    '''
    Part A:
    First we need to build the x^2 + p^2 ground state.
    '''
    qc = build_circuit_gs_prep(num_qubits, x_max, 60, 600)
    '''
    Part B:
    Now go to lambda * x^4 ground state with adiabatic state prep
    '''
    dt = tPrep / 2 / nSteps
    for j in range(1, 2 * nSteps + 1):
        u = (j - 0.5) / nSteps
        lam_t = lamOf(u, lam)
        dm_ctr = dm2Of(u, lam)
        strang_step(qc, lam_t, dm_ctr, dt, x_max)
    return qc

In [ ]:
# Part A
def build_circuit_gs_prep(num_qubits, x_max, tSteps = 60, nSteps = 600):
    # GS is just exp( - x ** 2 / 2)
    dx = 2 * x_max / (2 ** num_qubits - 1)
    x_values = np.linspace(-x_max, x_max, 2 ** num_qubits)
    gs = np.exp( - x_values ** 2 / 2)
    gs /= np.sqrt(np.sum(np.abs(gs)**2))
    qc = QuantumCircuit(num_qubits)
    qc.initialize(gs)
    return qc

In [116]:
x_max = 10
num_qubits = 5

In [117]:
lam = 5

In [ ]:
qc = build_circuit(lam, num_qubits, x_max)

In [ ]:
state = Statevector.from_instruction(qc)

# Check with numpy

In [ ]:
def jlp_grids(n_qubits: int, x_max: float):
    N  = 2**n_qubits
    dx = 2.0 * x_max / (N - 1)
    j  = np.arange(N, dtype=float)
    # position grid (exactly N samples)
    x_vals = -x_max + j * dx
    # half-shifted momentum grid
    k_vals = -np.pi/dx + (j + 0.5) * (2.0*np.pi)/(N * dx)
    return x_vals, k_vals, dx

def plane_wave_S(n_qubits: int, x_max: float):
    x_vals, k_vals, _ = jlp_grids(n_qubits, x_max)
    N = 2**n_qubits
    S = np.exp(1j * np.outer(k_vals, x_vals)) / np.sqrt(N)
    return S
def jlp_X_P(n_qubits: int, x_max: float):
    x_vals, k_vals, _ = jlp_grids(n_qubits, x_max)
    X = np.diag(x_vals.astype(complex))
    S = plane_wave_S(n_qubits, x_max)
    P = S.conj().T @ np.diag(k_vals.astype(complex)) @ S
    return X, P

def jlp_H(n_qubits: int, x_max: float, lam: float, m: float):
    X, P = jlp_X_P(n_qubits, x_max)
    X2 = X @ X
    return 0.5*(P@P) + 0.5*(1.0+m)*X2 + lam*(X2@X2)

def exact_diagonalize(n_qubits: int, x_max: float, lam: float, m: float):
    H = jlp_H(n_qubits, x_max, lam, m)
    evals, evecs = la.eigh(H)   # columns are eigenvectors
    return evals, evecs, H
if __name__ == "__main__":
    n_q   = num_qubits
    x_max = 10
    # lam   = 0.0  # no lambda here we are looking at the SHO ground state
    m     = 0.0

    evals, evecs, H = exact_diagonalize(n_q, x_max, lam, m)
    print("Lowest 6 eigenvalues:", np.round(evals[:6], 8))

    # compare your Gaussian initializer to the discrete ground state if you want:
    N = 2**n_q
    x_vals, _, _ = jlp_grids(n_q, x_max)
    gaussian = np.exp(-0.5 * x_vals**2).astype(complex)
    gaussian /= np.linalg.norm(gaussian)

    v0 = evecs[:, 0]
    overlap = float(abs(np.vdot(v0, state))**2)
    print("Overlap(prepared state, discrete ground):", overlap)